In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [ ]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

#estimated salary is my output feature my ouptut and rest is independent features
#and hence this is a regression problem

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [5]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [7]:
geo_encoder = OneHotEncoder()
geo_encoded = geo_encoder.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=geo_encoder.get_feature_names_out(['Geography']))
data = pd.concat([data, geo_encoded_df], axis=1)
data = data.drop('Geography', axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [8]:
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']



In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
#save all the   pickle file
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

with open('geo_encoder.pkl', 'wb') as f:
    pickle.dump(geo_encoder, f) 

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('columns.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)



ANN WITH REGRESSION PROBLEM STATEMENT 


In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [14]:
#build the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1) #output layer for regression
])

#when you don't give any activation fcuntion its linear by def . also for 
#regression linear activation function is used 

#compile the model

model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [15]:
#logs
from keras.callbacks import TensorBoard, EarlyStopping
import datetime

logs_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=logs_dir, histogram_freq=1)

In [16]:
#early stopping
#we use validation loss this time as the stopping condition
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [17]:
#train the model
history = model.fit(
    X_train, y_train,
    validation_data = (X_test, y_test),
    epochs= 100,
    callbacks = [tensorboard_callback, early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 0s 730us/step - loss: 100367.2969 - mae: 100367.2969 - val_loss: 98481.5703 - val_mae: 98481.5703
Epoch 2/100
250/250 [==============================] - 0s 419us/step - loss: 99505.7812 - mae: 99505.7812 - val_loss: 96741.4922 - val_mae: 96741.4922
Epoch 3/100
250/250 [==============================] - 0s 408us/step - loss: 96501.2422 - mae: 96501.2422 - val_loss: 92350.8828 - val_mae: 92350.8828
Epoch 4/100
250/250 [==============================] - 0s 418us/step - loss: 90687.2422 - mae: 90687.2422 - val_loss: 85153.3750 - val_mae: 85153.3750
Epoch 5/100
250/250 [==============================] - 0s 408us/step - loss: 82367.4766 - mae: 82367.4766 - val_loss: 76099.0938 - val_mae: 76099.0938
Epoch 6/100
250/250 [==============================] - 0s 415us/step - loss: 72837.7031 - mae: 72837.7031 - val_loss: 66891.7422 - val_mae: 66891.7422
Epoch 7/100
250/250 [==============================] - 0s 421us/step - loss: 63950.7070 - ma

In [ ]:
%load_ext tensorboard
%tensorboard --logdir regressionlogs/fit




The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6009 (pid 56604), started 0:00:04 ago. (Use '!kill 56604' to kill it.)

In [21]:
!kill 56604'


zsh:1: unmatched '


In [22]:

test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss}, Test MAE: {test_mae}')


63/63 [==============================] - 0s 217us/step - loss: 50297.3359 - mae: 50297.3359
Test Loss: 50297.3359375, Test MAE: 50297.3359375


In [23]:
model.save('salary_regression_model.h5')

/Users/kshitoki/Documents/annclassification/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
